In [1]:
# importing dataset from the local pc
from google.colab import files
uploaded = files.upload()

Saving orders_bronze_silver_gold_1000_rows.csv to orders_bronze_silver_gold_1000_rows.csv


In [2]:
# creating spark session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Layers_parctise")
    .master("local[*]")
    .getOrCreate()
)

In [3]:
# printing spark version
print("spark version : ", spark.version)
print("Spark seeion created successfully")

spark version :  4.0.4
Spark seeion created successfully


In [4]:
source_path = "/content/orders_bronze_silver_gold_1000_rows.csv"

# reading the raw file and creating a dataframe
raw_df = (spark.read.option("header", True).option("inferSchema", True).csv(source_path))

In [5]:
print("Total rows : ", raw_df.count())
print(raw_df.printSchema())

Total rows :  1020
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)

None


In [6]:
# dealing with the paths of all three layers
from pathlib import Path

BASE_PATH = "/content/orders_pipeline"

BRONZE_PATH = f"{BASE_PATH}/bronze/orders"
SILVER_PATH = f"{BASE_PATH}/silver/orders"
QUARANTINE_PATH = f"{BASE_PATH}/quarantine/orders"
GOLD_PATH = f"{BASE_PATH}/gold/daily_sales"

#creating folders to store data of every layer
Path(BASE_PATH).mkdir(parents=True, exist_ok=True)

print("Pipeline output paths created:")
print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)
print("Quarantine:", QUARANTINE_PATH)
print("Gold:", GOLD_PATH)

Pipeline output paths created:
Bronze: /content/orders_pipeline/bronze/orders
Silver: /content/orders_pipeline/silver/orders
Quarantine: /content/orders_pipeline/quarantine/orders
Gold: /content/orders_pipeline/gold/daily_sales


**BRONZE LAYER**

In [7]:
# adding metadata into the bronze layer
# Make sure to keep the remaining data as it is

# first col -> source_file -> default will be "orders_bronze_silver_gold_1000_rows.csv"
# second col ->  ingested_at -> will be timestamp of ingestion



In [9]:
from pyspark.sql import functions as F


bronze_df = (
    raw_df
    .withColumn("_source_file", F.lit("orders_bronze_silver_gold_1000_rows.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
)

In [10]:
# printing stastics of bronze layer data

print("Bronze row count:", bronze_df.count())
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

Bronze row count: 1020
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)

+--------+-----------+-------+----------+----------+--------------+---------------------------------------+--------------------------+
|order_id|customer_id|amount |order_date|status    |payment_method|_source_file                           |_ingested_at              |
+--------+-----------+-------+----------+----------+--------------+---------------------------------------+--------------------------+
|50001   |1235       |1072.0 |2026-09-29|completed |Card          |orders_bronze_silver_gold_1000_rows.csv|2026-10-15 07:22:30.596152|
|50002   |1210       |4121.84|2026-09-02| PENDING  |UPI           |orders_bronze_silver_g

In [11]:
# writing this bronze data into a parquet file

(
    bronze_df.write
    .mode("overwrite")
    .parquet(BRONZE_PATH)
)

print("Bronze data is written succesfully into the parquet file")

Bronze data is written succesfully into the parquet file


In [12]:
# validating the records read from the parquet file
bronze_read_df = spark.read.parquet(BRONZE_PATH)

source_count = raw_df.count()
bronze_count = bronze_read_df.count()

print("Source row count:", source_count)
print("Bronze row count:", bronze_count)

assert source_count == bronze_count, "Bronze row-count validation failed!"

print("Bronze validation passed!")
bronze_read_df.show(5, truncate=False)

Source row count: 1020
Bronze row count: 1020
Bronze validation passed!
+--------+-----------+-------+----------+----------+--------------+---------------------------------------+-------------------------+
|order_id|customer_id|amount |order_date|status    |payment_method|_source_file                           |_ingested_at             |
+--------+-----------+-------+----------+----------+--------------+---------------------------------------+-------------------------+
|50001   |1235       |1072.0 |2026-09-29|completed |Card          |orders_bronze_silver_gold_1000_rows.csv|2026-10-15 07:24:26.71445|
|50002   |1210       |4121.84|2026-09-02| PENDING  |UPI           |orders_bronze_silver_gold_1000_rows.csv|2026-10-15 07:24:26.71445|
|50003   |1135       |4142.37|2026-09-26| PENDING  |Wallet        |orders_bronze_silver_gold_1000_rows.csv|2026-10-15 07:24:26.71445|
|50004   |1177       |4901.96|2026-10-11|Unknown   |NULL          |orders_bronze_silver_gold_1000_rows.csv|2026-10-15 07:24:

**SILVER LAYER**

In [14]:
# checking data quality
valid_statuses = ["COMPLETED", "PENDING", "CANCELLED"]

invalid_condition = (
    F.col("order_id").isNull()
    | F.col("customer_id").isNull()
    | F.col("amount").isNull()
    | (F.col("amount") < 0)
    | F.col("order_date").isNull()
    | F.col("status").isNull()
    | (~F.col("status").isin(valid_statuses))
)

valid_candidates_df = standardized_df.filter(~invalid_condition)

invalid_df = (
    standardized_df
    .filter(invalid_condition)
    .withColumn("_reject_reason", F.lit("INVALID_DATA_OR_BUSINESS_RULE"))
)

print("Standardized rows:", standardized_df.count())
print("Valid candidates:", valid_candidates_df.count())
print("Initially invalid rows:", invalid_df.count())

Standardized rows: 1020
Valid candidates: 871
Initially invalid rows: 149


In [17]:
# detecting and quaranting duplicate order_ids
# detecting the order_ids which are duplicate
duplicate_keys_df = (
    valid_candidates_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

# joining both dfs to create duplicate datframe
duplicate_rows_df = (
    valid_candidates_df
    .join(duplicate_keys_df, on="order_id", how="inner")
    .withColumn("_reject_reason", F.lit("DUPLICATE_ORDER_ID"))
)

In [18]:
# creating silver layer without the duplicate order_ids
silver_df = valid_candidates_df.join(
    duplicate_keys_df,
    on="order_id",
    how="left_anti"
)


In [19]:
print("Duplicate order IDs:", duplicate_keys_df.count())
print("Rows quarantined as duplicates:", duplicate_rows_df.count())
print("Final Silver rows:", silver_df.count())

Duplicate order IDs: 18
Rows quarantined as duplicates: 36
Final Silver rows: 835


In [20]:
# creating quarantine dataset
quarantine_df = invalid_df.unionByName(duplicate_rows_df)

print("Quarantine rows:", quarantine_df.count())
quarantine_df.groupBy("_reject_reason").count().show()

Quarantine rows: 185
+--------------------+-----+
|      _reject_reason|count|
+--------------------+-----+
|INVALID_DATA_OR_B...|  149|
|  DUPLICATE_ORDER_ID|   36|
+--------------------+-----+



In [21]:
# validating after the silver layer
input_count = standardized_df.count()
silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

print("Input rows:", input_count)
print("Silver accepted rows:", silver_count)
print("Quarantine rejected rows:", quarantine_count)

assert input_count == silver_count + quarantine_count, (
    "Row-count reconciliation failed!"
)

print("Silver row-count reconciliation passed!")

Input rows: 1020
Silver accepted rows: 835
Quarantine rejected rows: 185
Silver row-count reconciliation passed!


In [22]:
# writing silver and quarantine layer data into respective parquet files
(
    silver_df.write
    .mode("overwrite")
    .parquet(SILVER_PATH)
)

(
    quarantine_df.write
    .mode("overwrite")
    .parquet(QUARANTINE_PATH)
)

print("Silver and quarantine outputs written successfully!")

Silver and quarantine outputs written successfully!


**GOLD LAYER**

In [24]:
# reading silver data
silver_read_df = spark.read.parquet(SILVER_PATH)


In [25]:
# for instance if our buisiness requirement includes only COMPLETED orders
# creating gold layer data
gold_df = (
    silver_read_df
    .filter(F.col("status") == "COMPLETED")
    .groupBy("order_date")
    .agg(
        F.count("*").alias("completed_orders"),
        F.sum("amount").alias("total_sales"),
        F.round(F.avg("amount"), 2).alias("average_order_value")
    )
    .orderBy("order_date")
)

gold_df.show(50, truncate=False)
gold_df.printSchema()

+----------+----------------+-----------+-------------------+
|order_date|completed_orders|total_sales|average_order_value|
+----------+----------------+-----------+-------------------+
|2026-09-01|7               |17342.69   |2477.53            |
|2026-09-02|11              |26494.00   |2408.55            |
|2026-09-03|9               |18338.21   |2037.58            |
|2026-09-04|8               |23174.78   |2896.85            |
|2026-09-05|13              |29417.01   |2262.85            |
|2026-09-06|9               |20190.82   |2243.42            |
|2026-09-07|6               |12319.39   |2053.23            |
|2026-09-08|10              |26969.62   |2696.96            |
|2026-09-09|11              |27530.61   |2502.78            |
|2026-09-10|8               |19251.29   |2406.41            |
|2026-09-11|7               |20122.15   |2874.59            |
|2026-09-12|9               |26219.43   |2913.27            |
|2026-09-13|7               |17066.57   |2438.08            |
|2026-09

In [26]:
# writing gold data into the parquet file
(
    gold_df.write
    .mode("overwrite")
    .parquet(GOLD_PATH)
)

print("Gold dataset written successfully!")

Gold dataset written successfully!


In [28]:
# reading data from the gold layer parquet file and validating it
gold_read_df = spark.read.parquet(GOLD_PATH)

print("Gold report rows:", gold_read_df.count())
gold_read_df.show(50, truncate=False)


# checking whether the parquet data count is same as dataframe count
assert gold_read_df.count() == gold_df.count()


assert gold_read_df.filter(
    F.col("completed_orders") <= 0
).count() == 0

assert gold_read_df.filter(
    F.col("total_sales").isNull()
).count() == 0

print("Gold validation passed!")

Gold report rows: 45
+----------+----------------+-----------+-------------------+
|order_date|completed_orders|total_sales|average_order_value|
+----------+----------------+-----------+-------------------+
|2026-09-01|7               |17342.69   |2477.53            |
|2026-09-02|11              |26494.00   |2408.55            |
|2026-09-03|9               |18338.21   |2037.58            |
|2026-09-04|8               |23174.78   |2896.85            |
|2026-09-05|13              |29417.01   |2262.85            |
|2026-09-06|9               |20190.82   |2243.42            |
|2026-09-07|6               |12319.39   |2053.23            |
|2026-09-08|10              |26969.62   |2696.96            |
|2026-09-09|11              |27530.61   |2502.78            |
|2026-09-10|8               |19251.29   |2406.41            |
|2026-09-11|7               |20122.15   |2874.59            |
|2026-09-12|9               |26219.43   |2913.27            |
|2026-09-13|7               |17066.57   |2438.08 